# M4：VideoSeal固定外部参考

随B分支发布的直接Run all入口。官方videoseal_1.0，原生256位共同owner消息、默认强度0.2/JND、chunk32/step4、FP32。当前仅完成实现与CPU检查，尚无真实M4结果。原B M3负结论保留。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. 一次尝试与云端预检

已有attempt/启动收据/执行痕迹即只读，云端未知不启动。quota只记录；完成准备后、worker前才写读普通启动收据，不是并发原子锁。失败或中断消费一次预算，不补跑。


In [ ]:

from pathlib import Path
from collections import Counter
import json,os,shutil,stat,subprocess,sys,tempfile,urllib.request,zipfile
os.environ.update(HF_HOME='/content/video-wm-m4/cache/huggingface',HF_HUB_CACHE='/content/video-wm-m4/cache/huggingface/hub',TORCH_HOME='/content/video-wm-m4/cache/torch')
INPUT=Path('/content/drive/MyDrive/Video-WM/B-Rank3R-New-Source-Confirmation-V1/fixed-four-five-v1')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-M4-VideoSeal-Reference-V1/fixed-four-five-v1')
SOURCE_REF='dev/local-joint-state-payload-v1'
SOURCE_DIRECTORY=None
"""Colab repeat storage: actual Drive API reads, separate from method evaluation.

No model, codec, production runtime or network work occurs at import. The
Notebook embeds this readable module for cloud guarding before source setup.
"""
import json
from pathlib import Path
import shutil
import tempfile
import time
import uuid
import zipfile

REPEAT_NAME='fixed-four-five-v1'
DRIVE_COMPONENTS=('Video-WM','B-M4-VideoSeal-Reference-V1',REPEAT_NAME)
DEFAULT_OUTPUT='/content/drive/MyDrive/'+'/'.join(DRIVE_COMPONENTS)
CORE_JSON=('result.json','evaluation.json','ledger.json','planned_outputs.json','plan.json','protocol.json',
           'physical_counts.json','physical_events.json',
           'blind_complete.json','post_truth_access.json','attempt.json','progress.json')
HOSTS=tuple('b-rank3r-m%02d'%n for n in range(1,5))
MEDIA=tuple(f'media/{h}/{arm}/{scope}/video.mp4' for h in HOSTS for arm in ('VideoSeal',)
            for scope in ('full','CROP','DELETE_LEFT_GAP','DELETE_RIGHT_GAP','REPEAT_LEFT_GAP','REPEAT_RIGHT_GAP'))
ARTIFACT_FOLDERS=('receipts','media','extractions','quality')
ROOT_ARTIFACTS=set(CORE_JSON)|{'inputs.json','model.json','release.json','B_reference.json','repeat_start.json','repeat_start_unknown.json'}
FOLDER_MIME='application/vnd.google-apps.folder'


def _json(stream):
    def invalid(value):raise ValueError('non-JSON numeric constant: '+value)
    return json.load(stream,parse_constant=invalid)


def _name(name):
    if not isinstance(name,str) or not name or name in ('.','..') or any(c in name for c in ('/','\\',':','\x00')):
        raise ValueError('unsafe Drive child name')
    return name


def _quoted(value):
    return "'"+str(value).replace('\\','\\\\').replace("'","\\'")+"'"


class DriveClient:
    """Small Drive v3 adapter. Writes only one ordinary startup receipt."""
    def __init__(self,session):self.session=session
    def _metadata(self,method,url,**kwargs):
        response=self.session.request(method,url,timeout=(20,120),**kwargs)
        try:
            response.raise_for_status();return response.json()
        finally:response.close()
    def quota(self):
        return self._metadata('GET','https://www.googleapis.com/drive/v3/about',params={'fields':'storageQuota'}).get('storageQuota',{})
    def children(self,parents):
        query='trashed = false and ('+' or '.join(_quoted(p)+' in parents' for p in parents)+')'
        token=None;answer=[]
        while True:
            params=dict(q=query,spaces='drive',pageSize=1000,fields='nextPageToken,incompleteSearch,files(id,name,mimeType,size,parents)')
            if token:params['pageToken']=token
            value=self._metadata('GET','https://www.googleapis.com/drive/v3/files',params=params)
            if value.get('incompleteSearch'):raise RuntimeError('Drive returned an incomplete directory listing')
            answer.extend(value.get('files',[]));token=value.get('nextPageToken')
            if not token:return answer
    def read_to(self,meta,destination):
        """Read alt=media through EOF. No media decoding and no in-memory video."""
        response=self.session.request('GET','https://www.googleapis.com/drive/v3/files/'+meta['id'],
            params={'alt':'media'},headers={'Accept-Encoding':'identity'},stream=True,timeout=(20,120))
        count=0
        try:
            response.raise_for_status()
            with Path(destination).open('wb') as stream:
                for block in response.iter_content(chunk_size=1024*1024):
                    if block:stream.write(block);count+=len(block)
            if count==0:raise IOError('empty cloud blob')
            expected=meta.get('size')
            if expected is not None and count!=int(expected):raise IOError(f'cloud read length {count} differs from listed bytes {expected}')
            return count
        finally:response.close()
    def create_start(self,parent_id,payload):
        """Exactly one create request. Unknown responses are never retried here."""
        boundary='m4-'+uuid.uuid4().hex
        metadata=json.dumps(dict(name='repeat_start.json',parents=[parent_id],mimeType='application/json')).encode()
        body=json.dumps(payload,ensure_ascii=False).encode()
        chunks=[b'--'+boundary.encode()+b'\r\nContent-Type: application/json; charset=UTF-8\r\n\r\n'+metadata,
                b'--'+boundary.encode()+b'\r\nContent-Type: application/json\r\n\r\n'+body,
                b'--'+boundary.encode()+b'--\r\n']
        return self._metadata('POST','https://www.googleapis.com/upload/drive/v3/files',
            params={'uploadType':'multipart','fields':'id,name,mimeType,size,parents'},
            headers={'Content-Type':'multipart/related; boundary='+boundary},data=b'\r\n'.join(chunks))


def authenticate_drive():
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    auth.authenticate_user()
    credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
    return DriveClient(AuthorizedSession(credentials))


def resolve_folder(client,components=DRIVE_COMPONENTS):
    parent='root'
    for name in components:
        matches=[v for v in client.children([parent]) if v.get('name')==name]
        if not matches:return None
        if len(matches)!=1 or matches[0].get('mimeType')!=FOLDER_MIME:
            raise ValueError('ambiguous or non-folder Drive path component: '+name)
        parent=matches[0]['id']
    return parent


def local_artifacts(output):
    output=Path(output);found=[]
    for name in ROOT_ARTIFACTS:
        for suffix in ('','.tmp'):
            p=output/(name+suffix)
            if p.exists() or p.is_symlink():found.append(name+suffix)
    for name in ARTIFACT_FOLDERS:
        folder=output/name
        if folder.is_dir():found.extend(p.relative_to(output).as_posix() for p in folder.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)


def cloud_artifacts(client,folder):
    if folder is None:return []
    pending=[(folder,'')];found=[]
    while pending:
        parent,prefix=pending.pop()
        for item in client.children([parent]):
            name=_name(item['name']);relative=prefix+name
            if not prefix:
                if name.removesuffix('.tmp') in ROOT_ARTIFACTS:found.append(relative)
                elif name in ARTIFACT_FOLDERS and item.get('mimeType')==FOLDER_MIME:pending.append((item['id'],name+'/'))
            elif item.get('mimeType')==FOLDER_MIME:pending.append((item['id'],relative+'/'))
            else:found.append(relative)
        if found:return found  # One actual execution trace is enough to prevent a second attempt.
    return found


def preflight(client,output,components=DRIVE_COMPONENTS):
    local=local_artifacts(output);value=dict(status='UNKNOWN',safe_to_start=False,local_artifacts=local,cloud_artifacts=[],folder_id=None,errors=[],quota=None)
    if client is None:
        value['errors'].append('Drive API authentication unavailable');return value
    try:value['quota']=client.quota()
    except Exception as error:value['quota_error']=f'{type(error).__name__}: {error}'
    try:
        value['folder_id']=resolve_folder(client,components)
        value['cloud_artifacts']=cloud_artifacts(client,value['folder_id'])
        value['status']='EXISTING_ATTEMPT' if local or value['cloud_artifacts'] else 'NO_EXECUTION_OBSERVED'
        value['safe_to_start']=value['status']=='NO_EXECUTION_OBSERVED'
    except Exception as error:value['errors'].append(f'{type(error).__name__}: {error}')
    return value


def reserve_start(client,output,components=DRIVE_COMPONENTS,*,attempts=6,pause=5,sleep=time.sleep):
    """Recheck after setup; only a visible, unique mount-created folder is used.

    A receipt is an ordinary once-attempt record, not atomic cross-session exclusion.
    Folder visibility reads may repeat; create and its response are never retried.
    """
    output=Path(output);folder=None
    for index in range(attempts):
        state=preflight(client,output,components)
        if not state['safe_to_start']:return dict(status='NOT_STARTED',preflight=state)
        folder=state['folder_id']
        if folder:break
        if index+1<attempts:sleep(pause)
    if not folder:return dict(status='NOT_STARTED',error='mount-created repeat folder is not yet visible via Drive API; no folder or receipt was created')
    payload=dict(status='STARTING_FIXED_REPEAT',repeat=REPEAT_NAME,output_components=list(components),created_unix=time.time(),
                 purpose='durable one-attempt startup receipt; M4 fixed VideoSeal external reference',atomic_mutex=False)
    try:
        created=client.create_start(folder,payload)
        if not isinstance(created,dict) or not isinstance(created.get('id'),str):raise ValueError('create returned no usable receipt id')
        with tempfile.TemporaryDirectory(prefix='m4-receipt-') as temporary:
            path=Path(temporary)/'receipt.json';client.read_to(created,path)
            with path.open(encoding='utf-8') as stream:readback=_json(stream)
        if not isinstance(readback,dict) or readback.get('status')!='STARTING_FIXED_REPEAT':raise ValueError('startup receipt is not a readable start record')
        return dict(status='START_RECEIPT_CLOUD_READ_BACK',file_id=created['id'],folder_id=folder,atomic_mutex=False)
    except Exception as error:
        record=dict(status='START_UNKNOWN_NO_WORKER',error=f'{type(error).__name__}: {error}',created_unix=time.time(),retry_create=False)
        # Preserve uncertainty locally if the write may have reached Drive. Never masquerade as runtime attempt.json.
        try:
            with (output/'repeat_start_unknown.json').open('x',encoding='utf-8') as stream:json.dump(record,stream,ensure_ascii=False)
        except Exception as local_error:record['local_record_error']=f'{type(local_error).__name__}: {local_error}'
        return record


def inventory(client,folder):
    """Directory metadata only for supplementary JSON; no per-file content GET."""
    files={};pending={folder:''};visited=set()
    while pending:
        batch=dict(list(pending.items())[:40])
        for ident in batch:pending.pop(ident);visited.add(ident)
        for item in client.children(list(batch)):
            parents=[p for p in item.get('parents',[]) if p in batch]
            if len(parents)!=1:raise ValueError('Drive file has ambiguous listed parent')
            relative=batch[parents[0]]+_name(item['name'])
            if item.get('mimeType')==FOLDER_MIME:
                if item['id'] in visited or item['id'] in pending:raise ValueError('repeated directory id')
                pending[item['id']]=relative+'/'
            else:files.setdefault(relative,[]).append(item)
    return files


def _is_json(name):return name.endswith('.json') or name.endswith('.json.tmp')


def _load_summary(path):
    try:
        with Path(path).open(encoding='utf-8') as stream:value=_json(stream)
        if not isinstance(value,dict):return dict(read_error='root record must be an object')
        return {k:value.get(k) for k in ('status','reference_status','method_status','contribution_status','stop') if k in value}
    except Exception as error:return dict(read_error=f'{type(error).__name__}: {error}')


def _core_state(name,value):
    """Existing final-state semantics only; no whole-file equality or digest."""
    if not isinstance(value,dict):raise ValueError('core JSON root must be an object')
    state={k:value.get(k) for k in ('status','reference_status','method_status','contribution_status') if k in value}
    if name=='plan.json':state['units']=sorted(value.get('units',{}))
    elif name=='ledger.json':
        state.update(slots=sorted(value),returned=sum(isinstance(v,dict) and v.get('status')=='RETURNED' and v.get('attempts')==1 for v in value.values()),
            finished=sum(isinstance(v,dict) and v.get('status') in ('RETURNED','FAILED','INTERRUPTED') for v in value.values()),spent=sum(isinstance(v,dict) and v.get('attempts')==1 for v in value.values()))
    elif name=='planned_outputs.json':
        for key in ('tasks','quality'):
            rows=value.get(key,{})
            state[key+'_count']=len(rows);state[key+'_returned']=sum(isinstance(v,dict) and v.get('status')=='RETURNED' for v in rows.values())
            state[key+'_finished']=sum(isinstance(v,dict) and v.get('status') in ('RETURNED','FAILED','INTERRUPTED') for v in rows.values())
        state['reader_slots_valid']=all(len(t.get('slots',[]))==256 for t in value.get('tasks',{}).values())
    elif name=='physical_counts.json':
        state['counts']={k:{n:v.get(n) for n in ('planned','started','returned','failed')} for k,v in value.items() if isinstance(v,dict)}
    elif name=='blind_complete.json':
        state.update({k:value.get(k) for k in ('tasks','all_bits')})
    elif name=='evaluation.json':
        state['task_count']=len(value.get('tasks',{}));state['quality_count']=len(value.get('quality',{}))
        state['quality_finished']=sum(isinstance(v,dict) and v.get('status') in ('SCORED','RETURNED','FAILED','INTERRUPTED') for v in value.get('quality',{}).values())
    elif name=='physical_events.json':state['progress']=sum((v.get('started') or 0)+(v.get('returned') or 0) for v in value.values() if isinstance(v,dict))
    return state


def _state_score(name,state):
    if name=='result.json':return (state.get('status')=='COMPLETED_FIXED_ATTEMPT',0)
    if name=='evaluation.json':return (state.get('status')=='POST_RECORDED',state.get('task_count',0)+state.get('quality_finished',0))
    if name=='ledger.json':return (False,state.get('finished',0)*2+state.get('spent',0))
    if name=='planned_outputs.json':return (False,state.get('tasks_finished',0)+state.get('quality_finished',0))
    if name=='physical_counts.json':return (False,sum((v.get('started') or 0)+(v.get('returned') or 0)+(v.get('failed') or 0) for v in state.get('counts',{}).values()))
    if name=='physical_events.json':return (False,state.get('progress',0))
    if name=='plan.json':return (False,len(state.get('units',[])))
    return (state.get('status')=='READ_BACK',0)


def _final_state_issues(states,local_states):
    expected=local_states.get('result.json',{}).get('status')=='COMPLETED_FIXED_ATTEMPT' or states.get('result.json',{}).get('status')=='COMPLETED_FIXED_ATTEMPT'
    issues=[]
    if local_states.get('result.json',{}).get('status') in ('COMPLETED_FIXED_ATTEMPT','TECHNICAL_UNRESOLVED','METHOD_NOT_SUPPORTED'):
        if states.get('result.json',{}).get('status')!=local_states['result.json']['status']:issues.append('cloud result differs from the mounted final computation state')
        for name in ('evaluation.json','ledger.json','planned_outputs.json','physical_counts.json','physical_events.json','plan.json','blind_complete.json'):
            if _state_score(name,local_states.get(name,{}))>_state_score(name,states.get(name,{})):issues.append('cloud progress is behind mounted final record: '+name)
    if not expected:return issues
    def require(condition,message):
        if not condition:issues.append(message)
    require(states.get('result.json',{}).get('status')=='COMPLETED_FIXED_ATTEMPT','cloud result is not the completed computation record')
    evaluation=states.get('evaluation.json',{})
    require(evaluation.get('status')=='POST_RECORDED' and evaluation.get('task_count')==48 and evaluation.get('quality_count')==28,'cloud evaluation is not final POST_RECORDED with 48 tasks and28 quality records')
    for field in ('reference_status','method_status','contribution_status'):
        local=local_states.get('evaluation.json',{})
        if local.get('status')=='POST_RECORDED' and field in local:require(local[field]==evaluation.get(field),'cloud/local final '+field+' differ')
    outputs=states.get('planned_outputs.json',{})
    require(outputs.get('tasks_count')==outputs.get('tasks_returned')==48 and outputs.get('quality_count')==outputs.get('quality_returned')==28 and outputs.get('reader_slots_valid') is True,'cloud planned_outputs still contains unfinished or absent fixed rows')
    units=states.get('plan.json',{}).get('units',[]);ledger=states.get('ledger.json',{})
    require(bool(units) and ledger.get('slots')==units and ledger.get('returned')==len(units),'cloud ledger is not returned for the saved fixed plan')
    expected_counts=dict(model_load=1,embed=4,quantize=4,codec_encode=24,codec_probe=48,codec_decode=48,edit=20,extract=48,quality=28)
    counts=states.get('physical_counts.json',{}).get('counts',{})
    require(set(counts)==set(expected_counts) and all(counts[k].get('planned')==counts[k].get('started')==counts[k].get('returned')==n and counts[k].get('failed')==0 for k,n in expected_counts.items()),'cloud physical_counts are not the completed fixed budget')
    blind=states.get('blind_complete.json',{})
    require(blind.get('status')=='READ_BACK' and blind.get('tasks')==48 and blind.get('all_bits')==12288,'cloud blind_complete is not final')
    # attempt STARTED and progress active=post are normal final files in the successful runner.
    return issues


def confirm_and_backup(client,output,backup_directory='/content',components=DRIVE_COMPONENTS,*,progress=None):
    """Read 12 core blobs and 24 media from cloud; back up all visible JSON + 24 MP4.

    Supplementary JSON content comes from the mounted result tree. Cloud-only
    supplementary JSON is listed as missing from backup, never fetched as a new
    per-file cloud verification gate. Missing data never invokes physical work.
    """
    output=Path(output);backup_directory=Path(backup_directory);backup_directory.mkdir(parents=True,exist_ok=True)
    report=dict(status='SAVING',computation=dict(mounted_result=_load_summary(output/'result.json'),mounted_evaluation=_load_summary(output/'evaluation.json')),
        cloud_confirmation=dict(status='UNCONFIRMED',required_core_json=len(CORE_JSON),required_media=len(MEDIA),readable_core_json=0,readable_media=0,
            files={},missing=[],errors=[],supplementary_json_content_verified=False),
        backup=dict(status='INCOMPLETE',scope='all JSON visible in mounted tree or cloud directory metadata, plus fixed 24 MP4',files={},missing=[],errors=[]),
        method_status_modified=False,physical_calls=0)
    cloud=report['cloud_confirmation'];backup=report['backup'];listing={};inventory_ok=False
    local_states={};cloud_states={};valid_local=set()
    if client is not None:
        try:
            folder=resolve_folder(client,components)
            if folder is None:cloud['errors'].append('repeat output folder missing in Drive API')
            else:listing=inventory(client,folder);inventory_ok=True
        except Exception as error:cloud['errors'].append(f'{type(error).__name__}: {error}')
    else:cloud['errors'].append('Drive API unavailable; mounted files cannot establish cloud saving')
    json_local={}
    try:
        if output.is_dir():
            for path in output.rglob('*'):
                if path.is_file() and _is_json(path.name):
                    if path.is_symlink():backup['errors'].append('symlink result file not followed: '+str(path));continue
                    json_local[path.relative_to(output).as_posix()]=path
    except Exception as error:backup['errors'].append(f'local JSON inventory: {type(error).__name__}: {error}')
    for name,path in json_local.items():
        if name in CORE_JSON:
            try:
                with path.open(encoding='utf-8') as stream:local_states[name]=_core_state(name,_json(stream))
                valid_local.add(name)
            except Exception as error:backup['errors'].append(dict(path=name,error='mounted core JSON: '+str(error)))
    expected_json=set(CORE_JSON)|set(json_local)|{name for name in listing if _is_json(name)}
    for name,entries in listing.items():
        if len(entries)>1 and (_is_json(name) or name in MEDIA):backup['errors'].append('ambiguous cloud path in backup inventory: '+name)
    cloud['supplementary_json_listed']=len([p for p in listing if _is_json(p) and p not in CORE_JSON])
    required=set(CORE_JSON)|set(MEDIA)
    stamp=time.strftime('%Y%m%dT%H%M%S',time.gmtime())+'-'+uuid.uuid4().hex[:8]
    prefix=backup_directory/('b-m4-'+REPEAT_NAME+'-'+stamp)
    temporary_zip=Path(str(prefix)+'-incomplete.zip');report_path=Path(str(prefix)+'-backup-report.json')
    try:
        with tempfile.TemporaryDirectory(prefix='m4-cloud-read-',dir=backup_directory) as directory:
            spool=Path(directory);cloud_paths={}
            for index,name in enumerate(sorted(required),1):
                candidates=listing.get(name,[])
                if len(candidates)!=1:
                    (cloud['missing'] if not candidates else cloud['errors']).append(name if not candidates else 'ambiguous cloud path: '+name)
                    continue
                meta=candidates[0];path=spool/str(index)
                try:
                    length=client.read_to(meta,path)
                    if name.endswith('.json'):
                        with path.open(encoding='utf-8') as stream:value=_json(stream)
                        cloud_states[name]=_core_state(name,value)
                        if name in ('result.json','evaluation.json'):report['computation']['cloud_'+name.removesuffix('.json')]={k:value.get(k) for k in ('status','reference_status','method_status','contribution_status','stop') if k in value}
                        cloud['readable_core_json']+=1
                    else:cloud['readable_media']+=1
                    cloud_paths[name]=path;cloud['files'][name]=dict(status='CLOUD_READ_TO_EOF',bytes=length,file_id=meta['id'])
                except Exception as error:cloud['errors'].append(dict(path=name,error=f'{type(error).__name__}: {error}'))
                if progress and (index%8==0 or index==len(required)):progress(f'Cloud blob reads: {index}/{len(required)}')
            cloud['final_state_issues']=_final_state_issues(cloud_states,local_states)
            if cloud['readable_core_json']==len(CORE_JSON) and cloud['readable_media']==len(MEDIA) and not cloud['missing'] and not cloud['errors'] and not cloud['final_state_issues']:
                cloud['status']='CORE_RESULTS_AND_24_MEDIA_CLOUD_READABLE'
            else:cloud['status']='CORE_OR_MEDIA_CLOUD_UNCONFIRMED'
            with zipfile.ZipFile(temporary_zip,'w',allowZip64=True) as archive:
                for name in sorted(expected_json|set(MEDIA)):
                    local=json_local.get(name) if _is_json(name) else output/name
                    candidate=cloud_paths.get(name);origin='Drive API full read'
                    if _is_json(name) and local is not None and local.is_file():
                        try:
                            if name not in valid_local:
                                with local.open(encoding='utf-8') as stream:_json(stream)
                                valid_local.add(name)
                            if name not in cloud_paths or _state_score(name,local_states.get(name,{}))>=_state_score(name,cloud_states.get(name,{})):
                                candidate=local;origin='mounted readable JSON; selected by saved progress semantics, no whole-file equality'
                        except Exception as error:
                            backup['errors'].append(dict(path=name,error='mounted JSON cannot be parsed: '+str(error)))
                            try:archive.write(local,'unparsed_local/'+name,compress_type=zipfile.ZIP_DEFLATED,compresslevel=3)
                            except Exception as raw_error:backup['errors'].append(dict(path=name,error='cannot preserve malformed original: '+str(raw_error)))
                    if candidate is None:
                        candidate=local;origin='mounted result file; cloud content not confirmed'
                    if candidate is None or not candidate.is_file() or candidate.is_symlink():backup['missing'].append(name);continue
                    try:
                        if name in MEDIA and name not in cloud_paths:
                            length=candidate.stat().st_size
                            if length<=0:raise IOError('empty mounted MP4')
                            entries=listing.get(name,[])
                            if len(entries)==1 and entries[0].get('size') is not None and length!=int(entries[0]['size']):raise IOError('mounted MP4 length differs from listed cloud bytes')
                        if _is_json(name) and candidate==local and name not in valid_local:raise ValueError('malformed local JSON is preserved separately, not a complete backup member')
                        archive.write(candidate,'results/'+name,compress_type=zipfile.ZIP_DEFLATED if _is_json(name) else zipfile.ZIP_STORED,compresslevel=3 if _is_json(name) else None)
                        backup['files'][name]=dict(source=origin,bytes=candidate.stat().st_size,cloud_path_readable=name in cloud_paths,backed_up_content_from_cloud=candidate==cloud_paths.get(name))
                    except Exception as error:backup['errors'].append(dict(path=name,error=f'{type(error).__name__}: {error}'))
                backup['json_count']=sum(_is_json(n) for n in backup['files']);backup['media_count']=sum(n in MEDIA for n in backup['files'])
                backup['cloud_directory_inventory_complete']=inventory_ok
                backup['status']='COMPLETE_BACKUP_OF_LISTED_JSON_AND_24_MEDIA' if inventory_ok and not backup['missing'] and not backup['errors'] and backup['media_count']==24 else 'INCOMPLETE_BACKUP'
                report['status']='STORAGE_CHECK_FINISHED';report['backup_zip']=str(prefix)+'-complete.zip' if backup['status']=='COMPLETE_BACKUP_OF_LISTED_JSON_AND_24_MEDIA' else str(temporary_zip);report['report_path']=str(report_path)
                archive.writestr('backup_report.json',json.dumps(report,ensure_ascii=False,indent=2),compress_type=zipfile.ZIP_DEFLATED)
        if backup['status']=='COMPLETE_BACKUP_OF_LISTED_JSON_AND_24_MEDIA':
            complete_zip=Path(str(prefix)+'-complete.zip');temporary_zip.rename(complete_zip);report['backup_zip']=str(complete_zip)
    except Exception as error:
        backup['status']='INCOMPLETE_BACKUP';backup['errors'].append(f'backup creation: {type(error).__name__}: {error}')
        report['status']='STORAGE_CHECK_INCOMPLETE';report['backup_zip']=str(temporary_zip) if temporary_zip.is_file() else None
    report['report_path']=str(report_path)
    try:report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    except OSError as error:report['report_write_error']=f'{type(error).__name__}: {error}'
    return report


def storage_summary(report):
    cloud=report['cloud_confirmation'];backup=report['backup']
    return dict(computation=report['computation'],cloud_save_status=cloud['status'],
        cloud_core_json=f"{cloud['readable_core_json']}/{cloud['required_core_json']}",cloud_media=f"{cloud['readable_media']}/{cloud['required_media']}",
        cloud_missing_count=len(cloud['missing']),cloud_error_count=len(cloud['errors']),cloud_final_state_issues=cloud.get('final_state_issues',[]),backup_status=backup['status'],
        backup_json_count=backup.get('json_count',0),backup_media_count=backup.get('media_count',0),backup_missing_count=len(backup['missing']),
        backup_errors=len(backup['errors']),backup_zip=report.get('backup_zip'),report_path=report.get('report_path'),
        method_status_modified=False,physical_calls=0)

HOSTS=tuple('b-rank3r-m0'+str(i) for i in range(1,5))

FIXTURES={'CROP':(0,None),'DELETE_LEFT_GAP':(8,49),'DELETE_RIGHT_GAP':(8,102),'REPEAT_LEFT_GAP':(-8,49),'REPEAT_RIGHT_GAP':(-8,102)}

SCOPES=('full',*FIXTURES)

ARMS=('OFF','VideoSeal')

def read(path):
    def invalid(value):raise ValueError('nonfinite JSON constant: '+value)
    return json.loads(Path(path).read_text(encoding='utf-8'),parse_constant=invalid)

def plan_units():
    units={'inputs':'input','model_load':'model_load'}
    for h in HOSTS:
        units[h+'/raw']='raw_input';units[h+'/embed']='embed';units[h+'/quantize']='quantize'
        units['quality/'+h+'/precodec/full']='quality'
        for scope in SCOPES:
            if scope!='full':units[h+'/'+scope+'/edit']='edit'
            units[h+'/VideoSeal/'+scope+'/encode']='codec_encode'
            for arm in ARMS:
                for operation,kind in [('probe','codec_probe'),('decode','codec_decode'),('extract','extract')]:units[f'{h}/{arm}/{scope}/{operation}']=kind
            units['quality/'+h+'/readback/'+scope]='quality'
    units.update(model_release='cleanup',blind_complete='readback',post='evaluation')
    return units

def execution_artifacts(output):
    output=Path(output);names=('attempt.json','plan.json','protocol.json','ledger.json','result.json','evaluation.json','planned_outputs.json',
        'physical_counts.json','physical_events.json','progress.json','blind_complete.json','post_truth_access.json','model.json','release.json','B_reference.json','inputs.json','repeat_start.json','repeat_start_unknown.json')
    found=[n+s for n in names for s in ('','.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for name in ('receipts','media','extractions','quality'):
        p=output/name
        if p.is_dir():found.extend(str(v.relative_to(output)) for v in p.rglob('*') if v.is_file() or v.is_symlink())
    return sorted(found)

def recover_ledger(output):
    output=Path(output);units=plan_units();ordered=list(units)
    ledger={slot:dict(slot=slot,kind=kind,status='NOT_ENTERED',attempts=0) for slot,kind in units.items()};errors={}
    def valid(row,slot,receipt=False):
        if not isinstance(row,dict) or row.get('slot')!=slot or row.get('kind')!=units[slot]:return False
        status=row.get('status');attempt=row.get('attempts')
        return type(attempt) is int and ((not receipt and status=='NOT_ENTERED' and attempt==0) or
            (status in ('STARTED','RETURNED','FAILED','INTERRUPTED') and attempt==1))
    try:
        saved=read(output/'ledger.json')
        if not isinstance(saved,dict):raise ValueError('ledger root must be an object')
        for slot,row in saved.items():
            if slot not in units:errors[slot]='unknown saved ledger slot';continue
            if valid(row,slot):ledger[slot]=row
            else:ledger[slot].update(status='UNRESOLVED_LEDGER',attempts=1);errors[slot]='invalid saved ledger status/attempt/address'
    except Exception as error:errors['ledger.json']=str(error)
    folder=output/'receipts'
    for path in sorted(folder.iterdir()) if folder.is_dir() else []:
        if not (path.name.endswith('.json') or path.name.endswith('.json.tmp')):continue
        name=path.name.removesuffix('.tmp').removesuffix('.json');key=str(path.relative_to(output))
        if not name.isdigit() or not 0<=int(name)<len(ordered):errors[key]='unknown numeric receipt address';continue
        slot=ordered[int(name)]
        try:
            row=read(path)
            if not valid(row,slot,receipt=True):raise ValueError('invalid receipt status/attempt/address')
            if path.name.endswith('.tmp'):raise ValueError('pending atomic receipt; attempt consumption unresolved')
            ledger[slot]=row
        except Exception as error:
            errors[key]=str(error);ledger[slot].update(status='UNRESOLVED_RECEIPT',attempts=1,error=str(error))
    return dict(ledger=ledger,counts=dict(Counter(r['status'] for r in ledger.values())),read_errors=errors,physical_reentry=False)
CLOUD_CLIENT=None
SOURCE_ROOT=None
PREPARATION_ERROR=None
try:
    try:
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
    except ModuleNotFoundError as error:
        if error.name not in ('google','google.auth','google.auth.transport','google.auth.transport.requests','requests'):raise
        subprocess.run([sys.executable,'-m','pip','install','google-auth','requests'],check=True)
    CLOUD_CLIENT=authenticate_drive()
except Exception as error:
    PREPARATION_ERROR=f'{type(error).__name__}: {error}'
    print('Drive API不可用，本次不启动:',PREPARATION_ERROR)
PREFLIGHT=preflight(CLOUD_CLIENT,OUTPUT)
CAN_START=PREFLIGHT['safe_to_start']
print('云端/挂载执行痕迹:',PREFLIGHT['status'],'实时quota:',PREFLIGHT.get('quota'))
print('具体问题:',PREFLIGHT.get('errors'),PREFLIGHT.get('quota_error'))


## 2. 可用源码与固定输入

直接下载普通B分支，无需手上传ZIP。4份OFF precodec RGB24和24个OFF MP4均为已有M3输入；缺失不替换、不重新生成。模型、官方源码、临时RGB和日志在/content；Drive仅必要JSON与24个WM MP4。


In [ ]:
if CAN_START:
    try:
        Path('/content/video-wm-m4').mkdir(parents=True,exist_ok=True)
        workspace=Path(tempfile.mkdtemp(prefix='source-',dir='/content/video-wm-m4')).resolve()
        if SOURCE_DIRECTORY is None:
            archive=workspace/'source.zip'
            url='https://codeload.github.com/RICHAAARC/SC-SSTW/zip/refs/heads/'+SOURCE_REF
            urllib.request.urlretrieve(url,archive)
            destination=workspace/'source';destination.mkdir()
            with zipfile.ZipFile(archive) as source:
                for member in source.infolist():
                    normalized=member.filename.replace('\\','/')
                    target=(destination/normalized).resolve()
                    if normalized.startswith('/') or any(':' in p for p in normalized.split('/')) or not target.is_relative_to(destination.resolve()) or stat.S_ISLNK(member.external_attr>>16):
                        raise ValueError('unsafe source archive entry: '+member.filename)
                source.extractall(destination)
            entries=list(destination.glob('*/experiments/wan_state_clock/b_m4_videoseal_reference_v1_run.py'))
            if len(entries)!=1:raise FileNotFoundError('RANK3 source entry unavailable: '+url)
            SOURCE_ROOT=entries[0].parents[2]
        else:
            original=Path(SOURCE_DIRECTORY).expanduser().resolve()
            if not (original/'experiments/wan_state_clock/b_m4_videoseal_reference_v1_run.py').is_file():
                raise FileNotFoundError('RANK3 entry missing in SOURCE_DIRECTORY')
            SOURCE_ROOT=workspace/'source'
            shutil.copytree(original,SOURCE_ROOT,ignore=shutil.ignore_patterns('.git','__pycache__'))
        OUTPUT.mkdir(parents=True,exist_ok=True)
        print('隔离源码目录:',SOURCE_ROOT)
        print('原始输入:',INPUT,'新输出:',OUTPUT)

    except Exception as error:
        CAN_START=False
        PREPARATION_ERROR=f'{type(error).__name__}: {error}'
        print('源码准备失败:',PREPARATION_ERROR)
else:
    print('只读或云端未知：跳过源码下载。')


## 3. 原生模型与固定调用

只补实际导入缺失，深层cfg探针不加载模型；复用ffmpeg媒体链。4embed、24encode、48probe/decode、20edit、48detect。预计8 embedder/248 extractor网络前向，另8个非网络JND块。embed antialias=True、detect=False；一次detect原生FP32帧均值后>0，不另调extract_message。


In [ ]:
if CAN_START:
    try:

        VENDOR=Path('/content/video-wm-m4/vendor/videoseal')
        VENDOR.parent.mkdir(parents=True,exist_ok=True)
        if not (VENDOR/'videoseal/utils/cfg.py').is_file():
            if VENDOR.exists():raise FileNotFoundError('Existing vendor directory lacks readable videoseal/utils/cfg.py')
            stage=Path(tempfile.mkdtemp(prefix='official-',dir='/content/video-wm-m4'))
            archive=stage/'source.zip'
            urllib.request.urlretrieve('https://codeload.github.com/facebookresearch/videoseal/zip/refs/heads/main',archive)
            destination=stage/'source';destination.mkdir()
            with zipfile.ZipFile(archive) as zipped:
                for member in zipped.infolist():
                    normalized=member.filename.replace('\\','/')
                    target=(destination/normalized).resolve()
                    if normalized.startswith('/') or any(':' in p for p in normalized.split('/')) or not target.is_relative_to(destination.resolve()) or stat.S_ISLNK(member.external_attr>>16):raise ValueError('unsafe official source archive path')
                zipped.extractall(destination)
            entries=list(destination.glob('*/videoseal/utils/cfg.py'))
            if len(entries)!=1:raise FileNotFoundError('available official VideoSeal cfg missing')
            shutil.move(str(entries[0].parents[2]),str(VENDOR))
        environment=os.environ.copy()
        environment['PYTHONPATH']=os.pathsep.join([str(SOURCE_ROOT),str(VENDOR)])
        LOG = Path('/content/video-wm-m4/setup.log')
        def command(args, label, check=True):
            print(label, flush=True)
            with LOG.open('a') as stream:
                stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
                completed = subprocess.run(args,cwd=VENDOR,env=environment,stdout=stream,stderr=subprocess.STDOUT)
            if completed.returncode and check:
                raise RuntimeError(label+' failed; see '+str(LOG))
            return completed.returncode
        def ensure_imports(probe, packages, label):
            import re
            mapping = {p.replace('-', '_'):p for p in packages}; mapping.update({'cv2':'opencv-python','pywt':'PyWavelets','skimage':'scikit-image','yaml':'PyYAML'})
            repaired = set()
            while True:
                offset = LOG.stat().st_size if LOG.exists() else 0
                if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                    return
                with LOG.open('rb') as stream:
                    stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
                missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
                if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                    action = 'remove incompatible optional torchao'
                    args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
                elif missing and missing.group(1).split('.')[0] in mapping:
                    package = mapping[missing.group(1).split('.')[0]]
                    action = 'install missing '+package
                    args = [sys.executable, '-m', 'pip', 'install', package]
                else:
                    raise RuntimeError(label+' actual import failure; see '+str(LOG))
                if action in repaired:
                    raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
                repaired.add(action)
                command(args, action)
        ensure_imports('import numpy,torch,torchvision; import videoseal.utils.cfg', ['numpy','torch','torchvision','pandas','omegaconf','einops','lpips','timm','pycocotools','tqdm','pytorch-msssim','tensorboard','calflops','transformers','safetensors','scipy','ffmpeg-python','av','huggingface-hub','ftfy','sentencepiece','accelerate'], 'official VideoSeal cfg and tensor imports (no model load)')
        import shutil
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            command(['apt-get','-qq','update'], 'Prepare media dependency')
            command(['apt-get','-qq','install','-y','ffmpeg'], 'Install FFmpeg')
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('ffmpeg/ffprobe unavailable; see '+str(LOG))

        probe="from pathlib import Path; import importlib; root=Path(__import__('sys').argv[1]).resolve(); names=['runtime.wan.b_m4_videoseal_reference_v1','runtime.wan.b_m4_videoseal_reference_storage_v1','experiments.wan_state_clock.b_m4_videoseal_reference_v1_run']; modules=[importlib.import_module(n) for n in names]; assert all(Path(m.__file__).resolve().is_relative_to(root) for m in modules); print(*[m.__file__ for m in modules],sep='\n')"
        subprocess.run([sys.executable,'-c',probe,str(SOURCE_ROOT)],cwd=VENDOR,env=environment,check=True)
        START_RECEIPT=reserve_start(CLOUD_CLIENT,OUTPUT)
        print('启动收据:',START_RECEIPT)
        if START_RECEIPT['status']=='START_RECEIPT_CLOUD_READ_BACK':
            args=[sys.executable,'-u','-c','from experiments.wan_state_clock.b_m4_videoseal_reference_v1_run import _notebook_reserved_main; raise SystemExit(_notebook_reserved_main())','--input',str(INPUT),'--output',str(OUTPUT)]
            with Path('/content/video-wm-m4/runner.log').open('a') as stream:
                worker=subprocess.Popen(args,cwd=VENDOR,env=environment,stdout=stream,stderr=subprocess.STDOUT)
                try:
                    while worker.poll() is None:
                        try:worker.wait(timeout=30)
                        except subprocess.TimeoutExpired:
                            try:
                                progress=json.loads((OUTPUT/'progress.json').read_text());print(progress.get('active'),progress.get('counts'),flush=True)
                            except (OSError,ValueError):print('同一worker运行中；详见/content/video-wm-m4/runner.log',flush=True)
                except BaseException:
                    worker.terminate()
                    try:worker.wait(timeout=10)
                    except subprocess.TimeoutExpired:worker.kill();worker.wait()
                    raise
            print('Runner exit:',worker.returncode,'保存目录:',OUTPUT)
        else:print('启动收据未读回，未运行worker。')

    except BaseException as error:
        PREPARATION_ERROR=f'{type(error).__name__}: {error}'
        print('准备或worker未完成，保留原记录:',PREPARATION_ERROR)
    CAN_START=False
else:
    print('只读或云端未知：跳过setup/vendor/model/worker。')


## 4. 结果口径

48提取记录或固定未决全部落盘回读后才消息后评，12288位全分母。28质量/4468帧/4440相邻对，无质量PASS。OFF字面匹配不是误检率；不定义DETECT/FPR/wrong-key。B32与VideoSeal256异容量分列，B只引用原M3三读法及已有full质量。


In [ ]:

if local_artifacts(OUTPUT):
    recovery=recover_ledger(OUTPUT)
    print('一次尝试记录:',recovery['counts'],'读取问题:',len(recovery['read_errors']))
    for name in ('result.json','evaluation.json'):
        try:value=json.loads((OUTPUT/name).read_text())
        except (OSError,ValueError) as error:
            print(name,'尚不可读:',type(error).__name__);continue
        if name=='result.json':print('计算状态:',value.get('status'),'固定分母:',value.get('denominators'),'调用:',value.get('physical_counts'),'内部:',value.get('observed_internal'),'停止:',value.get('stop'))
        else:print('外部参考状态:',value.get('reference_status'),'原生256位摘要:',value.get('summary'))
print('完整原始logits、固定未决、质量和B历史负结果:',OUTPUT)


## 5. 保存确认与本地备份

Google Drive API完整读取核心JSON与24个WM MP4，计算状态和保存状态分开。备份包含本次全部可读JSON和24媒体，缺失明确列出；发起下载不代表用户电脑已保存。保存检查不触发模型或codec。


In [ ]:

STORAGE_REPORT=confirm_and_backup(CLOUD_CLIENT,OUTPUT,backup_directory='/content',progress=lambda text:print(text,flush=True))
print(json.dumps(storage_summary(STORAGE_REPORT),ensure_ascii=False,indent=2))
backup_path=STORAGE_REPORT.get('backup_zip')
if backup_path and Path(backup_path).is_file() and zipfile.is_zipfile(backup_path):
    print('本地备份已生成:',backup_path)
    try:
        from google.colab import files
        files.download(backup_path)
        print('已发起下载，请确认浏览器下载完成。')
    except Exception as error:print('可在Colab文件面板下载:',backup_path,type(error).__name__,str(error))
